# 06a — Prepare real default dataset (Lending Club)

Build a **memory-conscious** training table from the raw accepted-loans CSV using **actual `loan_status` outcomes** as `default_risk`. **No model fitting** — data preparation only.

**Source:** `datasets/raw/lendingclub/accepted_2007_to_2018Q4.csv` (auto-resolved)  
**Output:** `datasets/processed/lendingclub_real_training_ready.csv`

`sklearn.model_selection.train_test_split` is used **only** for stratified row sampling (not training a predictor).

## 1. Load data safely (`usecols`)

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

REQUIRED_COLUMNS = [
    "loan_status",
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "dti",
    "delinq_2yrs",
    "fico_range_low",
    "fico_range_high",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "application_type",
]


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for directory in (cwd, *cwd.parents):
        if (directory / "backend").is_dir() and (directory / "ml").is_dir():
            return directory
    return cwd


def resolve_accepted_csv(lc_dir: Path) -> Path:
    if not lc_dir.is_dir():
        raise FileNotFoundError(
            f"Lending Club raw directory not found: {lc_dir.resolve()}"
        )
    canonical = lc_dir / "accepted_2007_to_2018Q4.csv"
    if canonical.is_file():
        return canonical.resolve()
    accepted = sorted(
        p.resolve()
        for p in lc_dir.glob("*.csv")
        if "accepted" in p.name.lower()
    )
    if not accepted:
        listing = sorted(p.name for p in lc_dir.glob("*.csv"))
        raise FileNotFoundError(
            f"No accepted CSV under {lc_dir.resolve()}. Files: {listing or '(none)'}"
        )
    return accepted[0]


REPO_ROOT = find_repo_root()
LC_DIR = REPO_ROOT / "datasets" / "raw" / "lendingclub"
DATA_PATH = resolve_accepted_csv(LC_DIR)

print(f"Resolved CSV: {DATA_PATH}")

try:
    df = pd.read_csv(DATA_PATH, usecols=REQUIRED_COLUMNS, low_memory=False)
except ValueError as exc:
    raise ValueError(
        "read_csv(usecols=...) failed — verify column names match this LC extract. "
        f"Required ({len(REQUIRED_COLUMNS)}): {REQUIRED_COLUMNS}"
    ) from exc

mem = int(df.memory_usage(deep=True).sum())
print(f"Loaded shape: {df.shape}")
print(f"Memory (deep estimate): {mem / (1024 ** 2):.2f} MiB ({mem:,} bytes)")

## 2. Real target (`default_risk`)

- **Safe (0):** `Fully Paid`, `Current`  
- **Risky (1):** `Charged Off`, `Default`, `Late (31-120 days)`, `Late (16-30 days)`  
- **Dropped:** ambiguous strings (`In Grace Period`, `Issued`, *Does not meet credit policy*, …) and **any other** `loan_status` not in the Safe/Risky sets.

In [ ]:
SAFE = {"Fully Paid", "Current"}
RISKY = {
    "Charged Off",
    "Default",
    "Late (31-120 days)",
    "Late (16-30 days)",
}
AMBIGUOUS_SUBSTR = (
    "In Grace Period",
    "Issued",
    "Does not meet credit policy",
    "Does not meet the credit policy",
)

raw = df["loan_status"].astype(str).str.strip()


def is_ambiguous(val: str) -> bool:
    v = val.strip()
    if v.lower() in {"nan", "none"}:
        return True
    return any(s in v for s in AMBIGUOUS_SUBSTR)


amb = raw.map(is_ambiguous)
print(f"Rows with ambiguous loan_status (dropped): {int(amb.sum()):,}")

df = df.loc[~amb].copy()
st = df["loan_status"].astype(str).str.strip()
keep = st.isin(SAFE) | st.isin(RISKY)
print(f"Rows with unmapped loan_status (dropped): {int((~keep).sum()):,}")
if (~keep).any():
    print(st.loc[~keep].value_counts().head(15).to_string())

df = df.loc[keep].copy()
st2 = df["loan_status"].astype(str).str.strip()
df["default_risk"] = st2.map(lambda x: 0 if x in SAFE else 1).astype("int8")

vc = df["default_risk"].value_counts().sort_index()
pct = df["default_risk"].value_counts(normalize=True).sort_index() * 100
print("\ndefault_risk distribution (full filtered data):")
print(pd.DataFrame({"count": vc, "percent": pct.round(4)}).to_string())

## 3. Stratified sample (max 200,000 rows)

Preserves **class proportions** using `train_test_split` on row indices (sampling utility only).

In [ ]:
from sklearn.model_selection import train_test_split

MAX_ROWS = 200_000
n = min(MAX_ROWS, len(df))
if n < len(df):
    idx = np.arange(len(df))
    sel, _ = train_test_split(
        idx,
        train_size=n,
        stratify=df["default_risk"].values,
        random_state=42,
        shuffle=True,
    )
    df = df.iloc[sel].reset_index(drop=True)
    print(f"Stratified sample applied: kept {n:,} / {MAX_ROWS:,} cap")
else:
    print(f"Row count ({len(df):,}) <= cap; no downsampling.")

print(f"Sampled shape: {df.shape}")
print(df["default_risk"].value_counts(normalize=True).round(4).to_string())

## 4. Clean features

In [ ]:
def parse_term_months(val) -> float:
    if pd.isna(val):
        return np.nan
    m = re.search(r"(\d+)", str(val))
    return float(m.group(1)) if m else np.nan


def parse_emp_length_years(val) -> float:
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    if s in {"", "nan", "n/a", "na"}:
        return np.nan
    if "<" in s or "less than" in s:
        return 0.0
    if "10+" in s or "10 +" in s:
        return 10.0
    m = re.search(r"(\d+)", s)
    return float(m.group(1)) if m else np.nan


def strip_percent_series(s: pd.Series) -> pd.Series:
    if s.dtype != object:
        return pd.to_numeric(s, errors="coerce")
    out = s.astype(str).str.replace("%", "", regex=False).str.strip()
    out = out.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return pd.to_numeric(out, errors="coerce")


df["term"] = df["term"].map(parse_term_months)
df["emp_length"] = df["emp_length"].map(parse_emp_length_years)

for col in ("int_rate", "revol_util"):
    if col in df.columns:
        df[col] = strip_percent_series(df[col])

for col in (
    "loan_amnt",
    "installment",
    "annual_inc",
    "dti",
    "delinq_2yrs",
    "fico_range_low",
    "fico_range_high",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2.0
df = df.drop(columns=["fico_range_low", "fico_range_high", "loan_status"])

print("Cleaned term (months), emp_length (years), numeric casts, fico_avg; dropped fico_* ranges and loan_status.")
print(f"Shape: {df.shape}")

## 5. Handle missing values

In [ ]:
def missing_table(frame: pd.DataFrame) -> pd.DataFrame:
    m = frame.isna().sum()
    p = (m / max(len(frame), 1) * 100).round(4)
    return (
        pd.DataFrame({"column": m.index, "missing_count": m.values, "missing_pct": p.values})
        .sort_values("missing_count", ascending=False)
        .reset_index(drop=True)
    )


print("=== Missing BEFORE imputation ===")
before = missing_table(df)
print(before.to_string())

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
cat_cols = [c for c in cat_cols if c != "default_risk"]

for c in num_cols:
    if c == "default_risk":
        continue
    med = df[c].median()
    if pd.isna(med):
        med = 0.0
    df[c] = df[c].fillna(med)

for c in cat_cols:
    mode = df[c].mode()
    fill = mode.iloc[0] if len(mode) else "unknown"
    df[c] = df[c].fillna(fill)

print("\n=== Missing AFTER imputation ===")
after = missing_table(df)
print(after.to_string())
print(f"\nTotal missing cells: {int(df.isna().sum().sum())}")

## 6. Encode features

- **Ordinal:** `grade` (A→1 … G→7), `sub_grade` (A1…G5 style)  
- **One-hot:** `home_ownership`, `verification_status`, `purpose`, `application_type`

In [ ]:
GRADE_ORDER = list("ABCDEFG")
CATS_ONEHOT = [
    "home_ownership",
    "verification_status",
    "purpose",
    "application_type",
]


def encode_grade(val) -> float:
    if pd.isna(val):
        return np.nan
    ch = str(val).strip().upper()[:1]
    if ch not in GRADE_ORDER:
        return np.nan
    return float(GRADE_ORDER.index(ch) + 1)


def encode_sub_grade(val) -> float:
    if pd.isna(val):
        return np.nan
    s = str(val).strip().upper()
    if len(s) < 2:
        return np.nan
    letter, digit = s[0], s[1]
    if letter not in GRADE_ORDER or not digit.isdigit():
        return np.nan
    return float(GRADE_ORDER.index(letter) * 5 + int(digit))


df["grade_encoded"] = df["grade"].map(encode_grade)
df["sub_grade_encoded"] = df["sub_grade"].map(encode_sub_grade)
df = df.drop(columns=["grade", "sub_grade"])

for col in CATS_ONEHOT:
    if col not in df.columns:
        print(f"Skip one-hot (missing): {col}")
        continue
    dummies = pd.get_dummies(df[col], prefix=col, dummy_na=False, dtype=np.int8)
    df = pd.concat([df.drop(columns=[col]), dummies], axis=1)
    print(f"One-hot: {col} -> +{dummies.shape[1]} columns")

for c in df.select_dtypes(include=[np.number]).columns:
    if c == "default_risk":
        continue
    med = df[c].median()
    if pd.isna(med):
        med = 0.0
    df[c] = df[c].fillna(med)

print(f"\nShape after encoding: {df.shape}")

## 7. Final validation

In [ ]:
feature_cols = [c for c in df.columns if c != "default_risk"]
mem_final = int(df.memory_usage(deep=True).sum())

print(f"Shape: {df.shape}")
print("\ndtypes:")
print(df.dtypes.to_string())
print("\nTarget distribution:")
print(df["default_risk"].value_counts().sort_index().to_string())
print(df["default_risk"].value_counts(normalize=True).round(4).sort_index().to_string())
print(f"\nMemory (deep estimate): {mem_final / (1024 ** 2):.2f} MiB ({mem_final:,} bytes)")
print(f"\nFeature count: {len(feature_cols)}")
print("Feature names:")
print(feature_cols)

## 8. Save output

In [ ]:
OUT = REPO_ROOT / "datasets" / "processed" / "lendingclub_real_training_ready.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

cols_order = feature_cols + ["default_risk"]
df[cols_order].to_csv(OUT, index=False)
print(f"Wrote: {OUT.resolve()}  |  shape: {df.shape}")